<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/treaper33-hue/ML-AI-assisted-/blob/main/compilers_and_triton/notebooks/b1_vector_add.ipynb)

# B1 — The first kernel, and the three ideas in it

**FUNDAMENTAL** · runtime a few seconds, mostly compilation

---

Vector addition, which is worth more attention than it usually gets. It
contains the three ideas you use in every kernel afterwards: you write
for a **block** rather than an element, `program_id` tells you which block
you are, and **masks** handle the ragged edge without branching.

> **This stage needs a GPU to verify its kernel.** In Colab: **Runtime ->
> Change runtime type -> T4 GPU**, then re-run the setup cell. Triton ships
> with Colab's PyTorch, so nothing needs installing.
>
> Without a GPU the module still runs and still teaches: the NumPy reference
> is checked and the kernel is reported as `SKIP`. **A skip is not a pass.**

---

## How to use this notebook

1. Run **Setup**. It defines the stage and runs nothing.
2. Run the stage and read its output, gates included.
3. Work through the sections below, which take the pieces apart and
   invite you to break them.

The gates are declared *before* the run. When one fails the useful question
is which is wrong — the code, or the threshold someone wrote down before the
problem was understood.

## Setup

Defines everything. Runs nothing.

In [ ]:
#@title Setup — run this cell first (collapsed: it is the whole stage)
# ===== embedded: track_b_triton/runtime.py =====
"""Shared runtime for the Triton track.

Triton compiles and runs on NVIDIA and AMD GPUs. Most people learning
it do not have one to hand, and official Triton wheels are Linux-only,
so every module in this track is built the same way:

1. a **reference implementation** in NumPy that states, unambiguously,
   what the kernel is supposed to compute;
2. a **Triton kernel** that computes it fast;
3. a **check** that runs the reference always and the kernel when a GPU
   is present.

Without a GPU the modules still run, still teach, and report the kernel
as `SKIPPED` rather than pretending. A skip is not a pass — the same
rule as the rest of this repository.

The fastest way to get a GPU for this track is Colab: Runtime → Change
runtime type → T4 GPU. Triton ships with PyTorch there, so nothing
needs installing.
"""

import dataclasses
import time
from typing import Callable, List, Optional

import numpy as np

try:
    import torch
    HAS_TORCH = True
except ImportError:                         # pragma: no cover
    HAS_TORCH = False

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:                         # pragma: no cover
    HAS_TRITON = False
    triton = None
    tl = None

HAS_GPU = bool(HAS_TORCH and torch.cuda.is_available())
CAN_RUN_KERNELS = bool(HAS_TRITON and HAS_GPU)

PASS = 'PASS'
FAIL = 'FAIL'
SKIP = 'SKIP'


def environment_banner() -> str:
    """Describes what this machine can actually verify.

    Returns:
        A multi-line description.
    """
    lines = ['environment:']
    if HAS_TORCH:
        lines.append(f'  torch    {torch.__version__}')
        if HAS_GPU:
            major, minor = torch.cuda.get_device_capability(0)
            lines.append(f'  gpu      '
                         f'{torch.cuda.get_device_name(0)} '
                         f'(sm_{major}{minor})')
        else:
            lines.append('  gpu      none visible')
    else:
        lines.append('  torch    not installed')
    lines.append(f'  triton   {"installed" if HAS_TRITON else "not installed"}')
    if CAN_RUN_KERNELS:
        lines.append('  -> kernels will RUN and be checked against the '
                     'reference')
    else:
        lines.append('  -> kernels will be SKIPPED; the reference still '
                     'runs and is checked')
        lines.append('     (Colab: Runtime -> Change runtime type -> T4 GPU)')
    return '\n'.join(lines)


@dataclasses.dataclass
class Result:
    """The outcome of one checked kernel.

    Attributes:
        name: What was checked.
        status: PASS, FAIL or SKIP.
        detail: Numbers a reader can judge.
        milliseconds: Kernel time, when it ran.
    """

    name: str
    status: str
    detail: str = ''
    milliseconds: Optional[float] = None

    def line(self) -> str:
        """Renders one report line."""
        timing = (f'  {self.milliseconds:7.3f} ms'
                  if self.milliseconds is not None else '')
        return f'  [{self.status}] {self.name:<40}{timing}  {self.detail}'


def relative_error(actual: np.ndarray, expected: np.ndarray) -> float:
    """Returns the largest relative elementwise difference.

    Absolute difference is the wrong measure for anything in float32:
    an error of 1e-3 is catastrophic near zero and invisible near 1e6.

    Args:
        actual: What the kernel produced.
        expected: What the reference produced.

    Returns:
        The maximum relative error.
    """
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    scale = np.maximum(np.abs(expected), 1e-6)
    return float(np.max(np.abs(actual - expected) / scale))


def normwise_error(actual: np.ndarray, expected: np.ndarray) -> float:
    """Returns ||actual - expected|| / ||expected||, Frobenius norm.

    `relative_error` above divides elementwise, which is the right
    measure when every expected value is bounded away from zero -- a
    softmax probability, a positive activation. It is the wrong measure
    the moment the output straddles zero, because it is then APE, and
    APE is unbounded at zeros: an absolute error of 1e-4 against an
    entry that happens to be 1e-6 reads as 1e+02 and says nothing about
    the computation.

    A matrix product of random data straddles zero everywhere. This is
    not a hypothetical: the elementwise measure reports 6.2e-02 for
    `torch.matmul` on one of B3's own test shapes, so a gate built on
    it rejects the reference implementation of the operation. Numerical
    linear algebra uses the normwise measure for exactly this reason,
    and so does B3.

    Args:
        actual: What the kernel produced.
        expected: What the reference produced.

    Returns:
        The normwise relative error.
    """
    actual = np.asarray(actual, dtype=np.float64)
    expected = np.asarray(expected, dtype=np.float64)
    scale = float(np.linalg.norm(expected.reshape(-1)))
    return float(np.linalg.norm((actual - expected).reshape(-1))
                 / max(scale, 1e-300))


def check(name: str, kernel: Callable[[], 'np.ndarray'],
          expected: np.ndarray, tolerance: float = 2e-2,
          repeats: int = 20, metric: str = 'elementwise') -> Result:
    """Runs a kernel, times it, and compares it with the reference.

    Args:
        name: What is being checked.
        kernel: Zero-argument callable returning the kernel's output.
        expected: The reference result.
        tolerance: Maximum acceptable relative error. The default is
            loose because float32 accumulation over a reduction is
            genuinely imprecise; each module tightens it where it can.
        repeats: Timed iterations after a warm-up.
        metric: 'elementwise' (the default) or 'normwise'. Choosing
            this per operation rather than once for the whole suite is
            deliberate -- see `normwise_error` for which is right when.

    Returns:
        The result record.

    Raises:
        ValueError: On an unknown metric name.
    """
    if metric not in ('elementwise', 'normwise'):
        raise ValueError(f'unknown metric {metric!r}')
    if not CAN_RUN_KERNELS:
        return Result(name, SKIP, 'no GPU or Triton in this runtime')
    try:
        output = kernel()                   # warm-up: compiles the kernel
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            output = kernel()
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - start) * 1000.0 / repeats

        array = (output.detach().float().cpu().numpy()
                 if HAS_TORCH and isinstance(output, torch.Tensor)
                 else np.asarray(output))
        if metric == 'normwise':
            error = normwise_error(array, expected)
            description = f'normwise relative error {error:.2e}'
        else:
            error = relative_error(array, expected)
            description = f'max relative error {error:.2e}'
        status = PASS if error < tolerance else FAIL
        return Result(name, status, description, elapsed)
    except Exception as error:              # noqa: BLE001 - reported
        return Result(name, FAIL, f'{type(error).__name__}: {error}')


def report(title: str, results: List[Result]) -> int:
    """Prints a result block and returns a process exit code.

    Args:
        title: Section heading.
        results: What to report.

    Returns:
        0 unless something failed. A skip is not a failure, but it is
        also not a pass, and the summary says so.
    """
    print(f'\n{title}')
    for result in results:
        print(result.line())
    failed = sum(r.status == FAIL for r in results)
    skipped = sum(r.status == SKIP for r in results)
    passed = sum(r.status == PASS for r in results)
    print(f'\n  passed {passed}   failed {failed}   skipped {skipped}')
    if skipped and not failed:
        print('  status: UNVERIFIED — the kernels did not run here')
    elif failed:
        print('  status: FAILED')
    else:
        print('  status: VERIFIED')
    return 1 if failed else 0


# ===== track_b_triton/b1_vector_add.py =====
"""B1 · FUNDAMENTAL — The first kernel, and the three ideas in it.

Vector addition is the "hello world" of GPU programming, and it is
worth more attention than it usually gets, because the three ideas it
contains are the three ideas you use in every kernel afterwards.

**1. You write the program for one block, not for one element.**
A CUDA kernel is written from the point of view of a single thread.
A Triton kernel is written from the point of view of a *block* of
elements, and the compiler decides how threads cover it. That single
change is why a competent Triton kernel is tens of lines where the CUDA
equivalent is hundreds — and why the compiler can vectorise and
schedule for you.

**2. `program_id` tells you which block you are.**
The grid says how many instances to launch; each instance asks which
one it is and computes its own slice of the data. There is no loop over
blocks — the grid *is* the loop, executed in parallel.

**3. Masks handle the edge.**
Data is rarely a multiple of the block size. Rather than branching, you
compute the full block and pass a boolean mask to the load and store so
out-of-range lanes read a default and write nothing. Branch-free edge
handling is the normal idiom, not a trick.

Run it:
    python b1_vector_add.py
"""

import sys
from typing import List

import numpy as np



if CAN_RUN_KERNELS:
    import torch
    import triton
    import triton.language as tl

    @triton.jit
    def vector_add_kernel(x_ptr, y_ptr, out_ptr, n_elements,
                          BLOCK_SIZE: tl.constexpr):
        """Adds two vectors, one block per program instance.

        Args:
            x_ptr: Pointer to the first input.
            y_ptr: Pointer to the second input.
            out_ptr: Pointer to the output.
            n_elements: Total length.
            BLOCK_SIZE: Elements per block; a compile-time constant, so
                the compiler can unroll and vectorise against it.
        """
        # Which block am I? The grid is the loop.
        pid = tl.program_id(axis=0)

        # The slice this instance owns.
        block_start = pid * BLOCK_SIZE
        offsets = block_start + tl.arange(0, BLOCK_SIZE)

        # The tail block runs past the end; mask rather than branch.
        mask = offsets < n_elements

        x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
        y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
        tl.store(out_ptr + offsets, x + y, mask=mask)

    def vector_add(x: 'torch.Tensor', y: 'torch.Tensor',
                   block_size: int = 1024) -> 'torch.Tensor':
        """Launches the kernel.

        Args:
            x: First input, on the GPU.
            y: Second input, same shape.
            block_size: Elements per program instance.

        Returns:
            The elementwise sum.

        Raises:
            ValueError: If the shapes differ.
        """
        if x.shape != y.shape:
            raise ValueError(f'shape mismatch: {x.shape} vs {y.shape}')
        output = torch.empty_like(x)
        n_elements = output.numel()
        # The grid is a function of the meta-parameters, so changing
        # BLOCK_SIZE changes the launch automatically.
        def grid(meta):
            """Instances needed for this block size."""
            return (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

        vector_add_kernel[grid](x, y, output, n_elements,
                                BLOCK_SIZE=block_size)
        return output


def reference_add(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """The specification: what the kernel must compute.

    Args:
        x: First input.
        y: Second input.

    Returns:
        The elementwise sum.
    """
    return x + y


def main() -> int:
    """Demonstrates the kernel and checks it against the reference.

    Returns:
        A process exit code.
    """
    print('=' * 70)
    print('B1 · FUNDAMENTAL — The first kernel')
    print('=' * 70)
    print()
    print(environment_banner())

    rng = np.random.default_rng(20260101)
    results: List[Result] = []

    # Sizes chosen so that one is an exact multiple of the block size
    # and the others are not: the masked tail is the part that breaks.
    for size in (1024, 100_000, 1_000_003):
        x = rng.standard_normal(size).astype(np.float32)
        y = rng.standard_normal(size).astype(np.float32)
        expected = reference_add(x, y)
        exact_blocks = size % 1024 == 0
        label = (f'n={size:,} '
                 f'({"exact blocks" if exact_blocks else "masked tail"})')

        if CAN_RUN_KERNELS:
            x_gpu = torch.from_numpy(x).cuda()
            y_gpu = torch.from_numpy(y).cuda()
            results.append(check(label,
                                 lambda a=x_gpu, b=y_gpu:
                                 vector_add(a, b),
                                 expected, tolerance=1e-6))
        else:
            results.append(Result(label, 'SKIP',
                                  'no GPU or Triton in this runtime'))

    # The reference is checked unconditionally, so this module still
    # verifies something on a CPU-only machine.
    small_x = rng.standard_normal(64).astype(np.float32)
    small_y = rng.standard_normal(64).astype(np.float32)
    reference_ok = np.allclose(reference_add(small_x, small_y),
                               small_x + small_y, atol=0)
    results.append(Result('reference matches numpy exactly',
                          'PASS' if reference_ok else 'FAIL',
                          'the specification is self-consistent'))

    code = report('Checks', results)

    print('\nWhat to take from this module')
    print('  * The grid is the loop. You never write a loop over blocks.')
    print('  * BLOCK_SIZE is a tl.constexpr, so it is known at compile')
    print('    time and the compiler can unroll and vectorise with it.')
    print('  * The mask is not an edge case bolted on; it is how every')
    print('    kernel handles a size that is not a multiple of the block.')
    print('  * Vector add is memory-bound: it reads 2N and writes N, and')
    print('    does one flop per element. No kernel can beat bandwidth,')
    print('    which is why fusion (B4) matters more than arithmetic.')
    return code

## Run it

The stage from start to finish, with its gates at the end.

In [ ]:
main()

## What the grid actually is

There is no loop over blocks. The grid *is* the loop, executed in
parallel, and `program_id` is how an instance finds out which iteration
it is. This is the arithmetic the launch does, in Python, so you can see
where the last partial block comes from.

In [ ]:
for n, block in ((1024, 256), (1000, 256), (1, 256),
                 (70000, 1024)):
    grid = -(-n // block)
    tail = n - (grid - 1) * block
    print(f'n={n:<7} BLOCK={block:<5} grid={grid:<4} '
          f'last block holds {tail} of {block} lanes')

## The mask is not a detail

Without a mask the final instance reads and writes past the end of the
buffer. On a GPU that is not a crash — it is silently wrong data, or
someone else's. The mask is what makes a block-shaped kernel safe on
arbitrary lengths, which is why every load and store in this repository
carries one.

In [ ]:
import numpy as np
n, block = 10, 4
for pid in range(-(-n // block)):
    offsets = pid * block + np.arange(block)
    mask = offsets < n
    print(f'pid={pid}  offsets={offsets}  mask={mask}')

## Ragged lengths, checked

Run the reference against lengths that are deliberately awkward. If a
kernel is going to be wrong about the edge, it will be wrong here — and
these are the sizes real tensors have.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
for n in (1, 2, 255, 256, 257, 1000, 4095):
    x, y = rng.standard_normal(n), rng.standard_normal(n)
    expected = reference_add(x, y)
    if CAN_RUN_KERNELS:
        import torch
        got = vector_add(torch.from_numpy(x).cuda(),
                         torch.from_numpy(y).cuda())
        gap = float((got.cpu().numpy() - expected).max())
        print(f'n={n:<6} kernel max error {gap:.2e}')
    else:
        print(f'n={n:<6} reference only (no GPU here)')

## Try it yourself

Change the block size and watch nothing break. The kernel is written for
a block of unknown width, so 64, 256 and 4096 are all correct — they
differ only in speed, which is the distinction A4 was about.

In [ ]:
if CAN_RUN_KERNELS:
    import numpy as np
    import torch
    x = torch.randn(1_000_003, device='cuda')
    y = torch.randn(1_000_003, device='cuda')
    for block in (64, 256, 1024, 4096):
        got = vector_add(x, y, block_size=block)
        ok = torch.allclose(got, x + y, atol=1e-6)
        print(f'BLOCK={block:<5} matches torch: {ok}')
else:
    print('needs a GPU: Runtime -> Change runtime type -> T4 GPU')

---

## Where this sits

The script version is `track_b_triton/b1_vector_add.py`, and
`compilers_and_triton/run_all.py` runs the whole ladder in order.
`compilers_and_triton/README.md` explains why the compiler track and the
Triton track are one course rather than two.